# Project Infinity-X 4.0 (Android 17) for POCO X3 Pro (vayu)
Automated Google Colab Builder for compiling Project Infinity-X 4.0 Android 17 on Google Cloud.

Output files are automatically saved to your **Google Drive** in `MyDrive/InfinityX_Vayu_Builds/`.

In [ ]:
# 1. Check System Hardware (CPU, RAM, Disk)
!lscpu | grep 'Model name\|CPU(s):'
!free -h
!df -h /

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/InfinityX_Vayu_Builds
!mkdir -p /content/drive/MyDrive/ccache

In [ ]:
# 3. Setup Build Tools and Dependencies
%%bash
sudo apt-get update -y
sudo apt-get install -y git-core gnupg flex bison build-essential zip curl zlib1g-dev \
    libc6-dev-i386 libncurses5 libncurses5-dev x11proto-core-dev libx11-dev \
    lib32z1-dev libgl1-mesa-dev libxml2 libxml2-utils xsltproc unzip fontconfig \
    python3 python-is-python3 aria2 rsync bc ccache

mkdir -p ~/bin
curl https://storage.googleapis.com/git-repo-downloads/repo > ~/bin/repo
chmod a+x ~/bin/repo
sudo ln -sf ~/bin/repo /usr/local/bin/repo

git config --global user.name 'PouyaFakhari'
git config --global user.email 'pouya@builder.local'
echo '[+] Tools installed successfully!'

In [ ]:
# 4. Initialize Source & Vayu Device Trees
%%bash
mkdir -p /content/infinity
cd /content/infinity

repo init -u https://github.com/ProjectInfinity-X/manifest.git -b 17 --depth=1 --git-lfs

mkdir -p .repo/local_manifests
cat << 'EOF' > .repo/local_manifests/vayu.xml
<?xml version="1.0" encoding="UTF-8"?>
<manifest>
  <project path="device/xiaomi/vayu" name="xetob/android_device_xiaomi_vayu" remote="github" revision="lineage-24.0" />
  <project path="device/xiaomi/sm8150-common" name="xetob/android_device_xiaomi_sm8150-common" remote="github" revision="lineage-24.0" />
  <project path="vendor/xiaomi/vayu" name="xetob/proprietary_vendor_xiaomi_vayu" remote="github" revision="lineage-24.0" />
  <project path="vendor/xiaomi/sm8150-common" name="xetob/proprietary_vendor_xiaomi_sm8150-common" remote="github" revision="lineage-24.0" />
  <project path="kernel/xiaomi/sm8150" name="xetob/android_kernel_xiaomi_sm8150" remote="github" revision="lineage-24.0" />
  <project path="hardware/xiaomi" name="LineageOS/android_hardware_xiaomi" remote="github" revision="lineage-24.0" />
</manifest>
EOF

repo sync -c -j$(nproc --all) --force-sync --no-clone-bundle --no-tags
echo '[+] Source sync completed!'

In [ ]:
# 5. Build Infinity-X 4.0 ROM
%%bash
cd /content/infinity
export CCACHE_DIR=/content/drive/MyDrive/ccache
export USE_CCACHE=1
ccache -M 30G

export WITH_GAPPS=false
source build/envsetup.sh
lunch lineage_vayu-userdebug || lunch infinity_vayu-userdebug

m bacon -j$(nproc --all)

In [ ]:
# 6. Copy Built ROM to Google Drive
%%bash
cp /content/infinity/out/target/product/vayu/*.zip /content/drive/MyDrive/InfinityX_Vayu_Builds/
cp /content/infinity/out/target/product/vayu/recovery.img /content/drive/MyDrive/InfinityX_Vayu_Builds/ || true
echo '=== SUCCESS! ROM COPIED TO YOUR GOOGLE DRIVE ==='
ls -lh /content/drive/MyDrive/InfinityX_Vayu_Builds/